# Spatial weight fields on the test functions

Companion notebook to *Understanding Hidden Spatial Preferences in Bayesian Optimization*. It uses the
`spatialbo` package only (`pip install -e ..`): the test functions of the paper, the three weight fields, and the
weighted GP-LCB rule with the package's surrogate and acquisition optimizer.

The acquisition is

$$\alpha(\mathbf{x}) = \mu_N(\mathbf{x}) + \lambda\,\frac{g(\mathbf{x};N,\ell_e)}{\bar g_N}\,\sigma_N(\mathbf{x};\ell_e),$$

for maximization of the standardized objective. $\mu_N$ is the posterior mean of a GP fitted to the data, $\sigma_N$
the posterior standard deviation at the exploration lengthscale $\ell_e$, and $g$ one of the three fields, divided by
its average $\bar g_N$ over the evaluated points. With $g = 1$ the rule is unweighted GP-LCB.

In [ ]:
import os, sys, math
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, torch
import matplotlib.pyplot as plt

from spatialbo import load_field, reference_lengthscale, FUNCS, standardized
from spatialbo.functions import MB4_POS, MB4_S, MB4_B
from spatialbo.acquisition import Surrogate, optimize

torch.set_default_dtype(torch.float64)

## 1. Test functions

`FUNCS` maps a name to `(f, D)`. Each `f` takes an `(M, D)` tensor in the unit cube and returns the value in
minimization form. `standardized(name)` returns the objective as the experiments see it, $h = -(f - m)/s$, with $m$
and $s$ the mean and standard deviation of $f$ over a fixed quasi-random sample.

In [ ]:
for name, (f, D) in FUNCS.items():
    print(f"{name:22s} D = {D}")

The functions `mb4_{D}d_{variant}` form the controlled landscape of Section 4.1: a weak periodic background with
Gaussian basins of identical width at a corner, a face and the interior of the cube (and an edge for $D \ge 3$). The
variant names the basin that is deepened, so the variants differ only in where the global minimum sits.

In [ ]:
D = 2
n = 161
grid = torch.linspace(0, 1, n)
U = torch.stack(torch.meshgrid(grid, grid, indexing="xy"), -1).reshape(-1, 2)

variants = list(MB4_POS[D])
fig, axes = plt.subplots(1, len(variants), figsize=(3.3 * len(variants), 3.2))
for ax, v in zip(axes, variants):
    f, _ = FUNCS[f"mb4_{D}d_{v}"]
    ax.imshow(f(U).reshape(n, n).numpy(), extent=[0, 1, 0, 1], origin="lower", cmap="viridis_r")
    for w, p in MB4_POS[D].items():
        ax.plot(*p, "o", ms=9 if w == v else 5, mfc="w" if w == v else "none", mec="w", mew=1.3)
    ax.set_title(f"mb4_{D}d_{v}"); ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
fig.suptitle(f"basin width s = {MB4_S[D]}, budget {MB4_B[D]} evaluations; the filled marker is the global minimum", y=1.02)
plt.show()

## 2. Weight fields

`load_field(kind, D)` returns the field `kind` in `{"geo", "uniform", "eff"}` for `D` in `2..6`. A field is a
function of the locations `X` (an `(M, D)` tensor in the unit cube), the number of observations `N` and the
exploration lengthscale `ell`, with `g = 1` at the center. The trained range is `N` in `[2D, 150]` and `ell` in
`[0.05, 3]`; inputs outside it are clamped. `reference_lengthscale(D)` is $\ell_e = 0.268\sqrt{D/2}$, the value used in
the paper.

In [ ]:
ell_e = reference_lengthscale(D)
fields = {k: load_field(k, D) for k in ("geo", "uniform", "eff")}

X = torch.tensor([[0.02, 0.02], [0.5, 0.02], [0.5, 0.5]])      # corner, face, center
for k, g in fields.items():
    print(f"g_{k:8s} N = 10:", g(X, 10, ell_e).numpy().round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
for ax, (k, g) in zip(axes, fields.items()):
    G = g(U, 10, ell_e).reshape(n, n).numpy()
    im = ax.imshow(np.minimum(G, 1.0), extent=[0, 1, 0, 1], origin="lower", cmap="magma", vmin=0.4, vmax=1.0)
    ax.set_title(f"g_{k}  (N = 10)"); ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
fig.colorbar(im, ax=axes, shrink=0.8, label="g(x)")
plt.show()

## 3. Weighted GP-LCB

`Surrogate(X, y, D, noise, ell_e)` fits the GP lengthscale for the mean and keeps a second factorization at `ell_e`
for the exploration term. `optimize` is the acquisition optimizer of the paper: 512 shared Sobol candidates, the 32
best plus 10 Boltzmann-sampled starts, one stacked L-BFGS-B with gradients through `g`. The rule names are those of
the result files.

| rule | exploration term | in the paper |
|---|---|---|
| `fit_fit` | $\sigma$ at the fitted lengthscale | standard GP-LCB |
| `fit_fix` | $\sigma$ at $\ell_e$ | unweighted, $g = 1$ |
| `fit_gfix` | $g_\text{geo}\,\sigma$ | geometric equalization |
| `fit_gufix` | $g_\text{uniform}\,\sigma$ | uniform allocation |
| `fit_gefffix` | $g_\text{eff}\,\sigma$ | efficiency-oriented exploration |

The loop below is that of `experiments/run_bo.py`: a Sobol initial design of $2D + 2$ points, the field divided by
its average over the evaluated points at every round, and the budget of the paper.

In [ ]:
RULES = {"fit_fit": None, "fit_fix": None, "fit_gfix": "geo", "fit_gufix": "uniform", "fit_gefffix": "eff"}
ATTR = {"geo": "g_ref", "uniform": "g_u", "eff": "g_eff"}
NOISE = 0.0025


def run_rule(func, rule, lam=2.0, seed=0):
    h, D = standardized(func)
    ell_e = reference_lengthscale(D)
    n0 = 2 * D + 2
    g = load_field(RULES[rule], D) if RULES[rule] else None

    Xo = torch.quasirandom.SobolEngine(D, scramble=True, seed=100 + seed).draw(n0).clamp(1e-3, 1 - 1e-3)
    yo = h(Xo) + math.sqrt(NOISE) * torch.randn(n0, generator=torch.Generator().manual_seed(1000 + seed))
    noise_gen = torch.Generator().manual_seed(8000 + seed)
    for t in range(MB4_B[D] - n0):
        N = len(Xo)
        sur = Surrogate(Xo, yo, D, NOISE, ell_e)
        if g is not None:
            c = float(g(Xo, N, ell_e).mean())
            setattr(sur, ATTR[RULES[rule]], lambda U, N=N, c=c: g(U, N, ell_e) / c)
        raw = torch.quasirandom.SobolEngine(D, scramble=True, seed=10000 * (seed + 1) + t).draw(512).clamp(1e-4, 1 - 1e-4)
        xn, _ = optimize(rule, sur, lam, D, raw, torch.Generator().manual_seed(500000 + 1000 * seed + t))
        yn = h(xn) + math.sqrt(NOISE) * torch.randn(1, generator=noise_gen)
        Xo, yo = torch.cat([Xo, xn]), torch.cat([yo, yn])
    return Xo, h(Xo)

Two placements of the global minimum in $D = 2$: in the corner, where the boundary preference of unweighted GP-LCB
helps, and in the interior, where it costs. All rules share the initial design and the candidate sets of the seed.

In [ ]:
funcs = [f"mb4_{D}d_corner", f"mb4_{D}d_interior"]
seed = 3
runs = {(fn, r): run_rule(fn, r, seed=seed) for fn in funcs for r in RULES}

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8), gridspec_kw={"width_ratios": [1.3, 1]})
for row, fn in zip(axes, funcs):
    f, _ = FUNCS[fn]
    h, _ = standardized(fn)
    h_star = h(torch.quasirandom.SobolEngine(D, scramble=True, seed=3).draw(2 ** 17)).max()
    for r in RULES:
        Xo, ho = runs[(fn, r)]
        regret = h_star - torch.cummax(ho, 0).values
        row[0].plot(np.arange(len(regret)) + 1, regret.clamp_min(1e-3).numpy(), label=r)
    row[0].axvline(2 * D + 2, color="0.6", ls=":", lw=1)
    row[0].set_yscale("log"); row[0].set_xlabel("evaluations"); row[0].set_ylabel("simple regret")
    row[0].set_title(fn); row[0].legend(fontsize=8)

    row[1].imshow(f(U).reshape(n, n).numpy(), extent=[0, 1, 0, 1], origin="lower", cmap="Greys_r", alpha=0.8)
    row[1].scatter(*runs[(fn, "fit_fix")][0][:2 * D + 2].T.numpy(), marker="x", s=30, c="k", label="initial design")
    for r in RULES:
        row[1].scatter(*runs[(fn, r)][0][2 * D + 2:].T.numpy(), s=14, label=r)
    row[1].set_xlim(0, 1); row[1].set_ylim(0, 1); row[1].set_aspect("equal")
    row[1].set_title("evaluations"); row[1].legend(fontsize=7, loc="center left", bbox_to_anchor=(1.02, 0.5))
fig.tight_layout()
plt.show()

With the optimum in the interior, the two unweighted rules settle in a local basin after visiting the faces, and the
three weighted rules reach the global basin with few evaluations near a wall. With the optimum in the corner, the
unweighted rules reach it because they look at the corners, and the weighted rules do not. A single seed on a single
function shows the mechanism, not the size of the effect; the paper reports many paired seeds per placement and
dimension.

## 4. A field in your own optimizer

The fields are differentiable in `X`. Evaluate them at the current number of observations and at the lengthscale
used for $\sigma$, and divide by the average over the evaluated points, so that the weighting changes where
exploration goes and not how much of it there is.

In [ ]:
Xq = torch.rand(5, D, requires_grad=True)
w = fields["geo"](Xq, 10, ell_e)
w.sum().backward()
print("g_geo:", w.detach().numpy().round(3))
print("dg/dx:\n", Xq.grad.numpy().round(3))